In [19]:
import collections as col
import pathlib as pl

import pandas as pd

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

glob_pattern = "**/*refOriented.haplotype*.fasta.gz"

search_path = CONFIG["local_hilbert_prefix"].joinpath(CONFIG["hilbert_verkko_assembly_folder"])
assert search_path.is_dir()

sample_annotation = CONFIG["project_repo"].joinpath("annotation", "norm", "samples_1kg_pedigree.ext.tsv")

samples = pd.read_csv(sample_annotation, sep="\t", header=0, comment="#")
samples.set_index("individual_id", inplace=True)

asm_norm = {
    "verkko-hi-c": "vrk-hic",
    "verkko-thic": "vrk-thc"
}

au_norm = {
    "haplotype1": "hap1",
    "haplotype2": "hap2"
}

norm_sample = {
    "HG002": "NA24385",
    "HG005": "NA24631"
}

rows = col.defaultdict(dict)
for fasta_file in search_path.glob(glob_pattern):
    sample = fasta_file.name.split(".")[0]
    sample = norm_sample.get(sample, sample)
    assembly_type = asm_norm[fasta_file.parent.parent.name]
    asm_unit = au_norm[fasta_file.name.split(".")[3]]
    try:
        sample_sex = samples.at[sample, "karyotype"]
    except KeyError:
        print("Missing: ", sample)
        continue
    rows[sample]["sample_sex"] = sample_sex
    rows[sample][f"asm_{asm_unit}"] = replace_path_prefix(fasta_file)
    rows[sample]["assembly_type"] = assembly_type

assemblies = pd.DataFrame.from_dict(rows, orient="index")
assemblies.index.name = "sample"
assemblies.sort_index(inplace=True)
assemblies = assemblies[["sample_sex", "assembly_type", "asm_hap1", "asm_hap2"]]

sample_sheet_file = CONFIG["project_repo"].joinpath("samples", "verkko_assemblies.tsv")
sample_sheet_file.parent.mkdir(exist_ok=True, parents=True)

with open(sample_sheet_file, "w") as table:
    _ = table.write(f"# {TIMESTAMP}\n")
    _ = table.write(f"# N={assemblies.shape[0]}\n")
    assemblies.to_csv(table, sep="\t", header=True, index=True)
    